Copyright 2026, Battelle Energy Alliance, LLC, ALL RIGHTS RESERVED

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
os.chdir("../../..")
from HelpingFunctions import ERCOTProcessor
from HelpingFunctions import WeatherProcessing
from HelpingFunctions import FeatureEngineering
from HelpingFunctions import ForecastingHelpers

import onnxruntime as ort
ort.set_default_logger_severity(4)

In [2]:
import os 
os.getcwd()

'/home/ortild/Amaranth/opensourcegridmodeling'

Data Preprocessing for RT Data - Setting Up Model Inputs

Linear Regression Model

In [3]:
import onnx
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/linear_no_mapie.onnx")

In [4]:
graph = onnx_model.graph
#print(onnx_model)
print(graph)
weights = onnx_model.graph.initializer[0].ListFields()[3][1][:]
interceptList = onnx_model.graph.initializer[1].ListFields()[3][1][:]

node {
  input: "X"
  input: "coef"
  output: "multiplied"
  name: "MatMul"
  op_type: "MatMul"
  domain: ""
}
node {
  input: "multiplied"
  input: "intercept"
  output: "resh"
  name: "Add"
  op_type: "Add"
  domain: ""
}
node {
  input: "resh"
  input: "shape_tensor"
  output: "variable"
  name: "Reshape"
  op_type: "Reshape"
  domain: ""
}
name: "ONNX(LinearRegression)"
initializer {
  dims: 13
  dims: 1
  data_type: 11
  name: "coef"
  double_data: 0.938745416511678
  double_data: -0.0029355835524287367
  double_data: -6.0131328821698735e-05
  double_data: -0.0017936289492126256
  double_data: -6.5064154666689841e-05
  double_data: -0.000911993588956408
  double_data: -0.0014550393514231315
  double_data: -0.0012154378839187086
  double_data: -5.2041704279304213e-18
  double_data: 0.0036475349789651021
  double_data: 0.0019535461581610255
  double_data: 0.0027213955768126121
  double_data: -0.022888753747047845
}
initializer {
  dims: 1
  data_type: 11
  name: "intercept"
  double

The onnx_model graph explains the operations performed on the inputs to receive the model output. 
The first node describes taking an input, X, and multiplying X with the model coefficients, coef, to produce a node output called multiplied. 
The second node describes taking multiplied and adding an intercept, intercept, to produce a node output called resh.
The third node describes reshaping the addition output to a single value for the output.

In [5]:
import random
index = 0
synthData = []
while index <= 100000:
    random_vector =  []
    for i in range(0,13):
        random_number = random.uniform(0, 1)
        random_vector.append(random_number)
    synthData.append(random_vector)
    index+=1

In [6]:
testX = synthData
input_df = pd.DataFrame(testX, columns=["inputA","inputB", "inputC", "inputD", "inputE", "inputF", "inputG", "inputH", "inputI", "inputJ", "inputK", "inputL", "inputM"])
print(input_df)

          inputA    inputB    inputC    inputD    inputE    inputF    inputG  \
0       0.842437  0.402726  0.258390  0.769353  0.489719  0.584422  0.083733   
1       0.105490  0.522960  0.914922  0.765311  0.630044  0.613666  0.258552   
2       0.197846  0.399911  0.470213  0.501213  0.784597  0.388274  0.113114   
3       0.561217  0.027720  0.738998  0.097708  0.681800  0.441053  0.026966   
4       0.908511  0.342775  0.416921  0.100341  0.224521  0.682388  0.736164   
...          ...       ...       ...       ...       ...       ...       ...   
99996   0.892271  0.328640  0.456362  0.346722  0.988736  0.719088  0.550117   
99997   0.031927  0.725395  0.488038  0.721794  0.033061  0.692458  0.469781   
99998   0.884125  0.593194  0.476666  0.815904  0.030766  0.467085  0.509926   
99999   0.926997  0.175723  0.843637  0.851973  0.818334  0.000547  0.061661   
100000  0.179332  0.226173  0.861873  0.283839  0.637481  0.500768  0.278298   

          inputH    inputI    inputJ   

In [7]:
import numpy as np
from sklearn.linear_model import LinearRegression
def predictLR(inputs):
    coef = np.array(weights)
    intercept = interceptList[0]
    model = LinearRegression(fit_intercept=False)
    model.coef_ = coef
    model.intercept_ = intercept
    manualX = np.array(inputs).reshape(1,-1)
    prediction = model.predict(manualX)
    return prediction


In [8]:
## test prediction 
final_prediction = []
for inputs in testX:
    output = predictLR(inputs)
    final_prediction.append(output)
output_df = pd.DataFrame(final_prediction, columns=["output"])
data_df = pd.concat([input_df, output_df], axis = 1)
data_df.head(5)

,inputA,inputB,inputC,inputD,inputE,inputF,inputG,inputH,inputI,inputJ,inputK,inputL,inputM,output
0,0.842437,0.402726,0.258390,0.769353,0.489719,0.584422,0.083733,0.853952,0.671058,0.458736,0.691437,0.469910,0.347095,0.784667
1,0.105490,0.522960,0.914922,0.765311,0.630044,0.613666,0.258552,0.569830,0.074863,0.049494,0.680056,0.583114,0.564479,0.086349
2,0.197846,0.399911,0.470213,0.501213,0.784597,0.388274,0.113114,0.196213,0.432928,0.879757,0.554683,0.935288,0.974848,0.169120
3,0.561217,0.027720,0.738998,0.097708,0.681800,0.441053,0.026966,0.179123,0.668656,0.527168,0.742708,0.378084,0.814773,0.513367
4,0.908511,0.342775,0.416921,0.100341,0.224521,0.682388,0.736164,0.016228,0.224406,0.811562,0.575405,0.551032,0.481556,0.846261


In [9]:
# Known model outputs
knownOut = [-0.117213, -0.11572873, -0.10967113, -0.0201042, -0.03743246, -0.0518272]


In [45]:
coef = np.array(weights)
intercept = interceptList[0]
invIntercept = (-1*intercept)
print(knownOut[0])
step1 = knownOut[0] + invIntercept
print(step1)
step2 = step1/coef
print(step2)
df = pd.DataFrame(step2)
print(df)
step3 = (step2*coef) +intercept
print(step3)

-0.117213
-0.11899045224823007
[-1.26754762e-01  4.05338326e+01  1.97884289e+03  6.63406176e+01
  1.82881731e+03  1.30472904e+02  8.17781678e+01  9.78992459e+01
  2.28644419e+16 -3.26221552e+01 -6.09099774e+01 -4.37240559e+01
  5.19864269e+00]
               0
0  -1.267548e-01
1   4.053383e+01
2   1.978843e+03
3   6.634062e+01
4   1.828817e+03
5   1.304729e+02
6   8.177817e+01
7   9.789925e+01
8   2.286444e+16
9  -3.262216e+01
10 -6.090998e+01
11 -4.372406e+01
12  5.198643e+00
[-0.117213 -0.117213 -0.117213 -0.117213 -0.117213 -0.117213 -0.117213
 -0.117213 -0.117213 -0.117213 -0.117213 -0.117213 -0.117213]


In [11]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

x = data_df["output"].to_numpy().reshape(-1,1)
y = data_df.drop(columns="output").to_numpy()
X_train, X_test, y_train, y_test = train_test_split(x, y, random_state=42, train_size = .8)

invModel = LinearRegression(fit_intercept=False)
invModel.intercept_ = invIntercept
invModel.fit(X_train, y_train)

y_pred = invModel.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae}")



Mean Absolute Error (MAE): 0.2910173846121527


In [46]:
knownOutput = [[-0.14729762]] #for testing?
predicted_vector = invModel.predict(knownOutput)
print(predicted_vector)

[[-0.1593608  -0.11887383 -0.11921568 -0.11889244 -0.11917069 -0.11906005
  -0.1189624  -0.11897445 -0.1192697  -0.11918272 -0.1195625  -0.11942143
  -0.11861831]]


In [28]:
full_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/full_data.csv', parse_dates=['time'], index_col=['time'])
hourly_res_norm = ForecastingHelpers.hourlyresiduals(full_df)

/home/ortild/Amaranth/opensourcegridmodeling/ElectricityDemandAustinTX/LoadForecastingAttacks/HelpingFunctions/ForecastingHelpers.py:74: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  hourly_res_norm['load'] = df_norm['load'].groupby(pd.Grouper(freq='M')).transform(lambda x: x - x.mean())


In [29]:
# train-validate-test split
train = hourly_res_norm[:'2014']
validate = hourly_res_norm['2015':'2016']
test = hourly_res_norm['2017':]

# setup training variables 
exog_tr = train.iloc[:,1:].values
ar_tr = train['load'].shift().bfill().values[:,None]
X_tr = np.hstack([ar_tr, exog_tr])
y_tr = train['load'].values

# setup validation variables
exog_val = validate.iloc[:,1:].values
y_val = validate['load'].values

# setup testing variables
exog_te = test.iloc[:,1:].values
ar_test = test['load'].shift().bfill().values[:,None]
y_test = test['load'].values
X_test = np.hstack([ar_test, exog_te])

# setup miscellaneous variables
yp_full = hourly_res_norm.loc[:'2016','load']
yp_val = hourly_res_norm.loc['2015':'2016','load']
yp_te = hourly_res_norm.loc['2017':,'load']
y_init_val = np.hstack([y_tr[-1], validate.iloc[167::168,0].values])
y_init_te = np.hstack([y_val[-1], test.iloc[167::168,0].values])

In [34]:
# Create test value
manualX = X_test[0]
print(manualX)

[-0.09924674  0.48484848  0.93        0.          0.          0.
  1.          0.          0.          0.          0.          0.
  1.        ]


In [33]:
# KNOWN OUTPUT FOR THIS CASE THAT WE ARE TRYING TO FIND THE INPUTS FOR
inputsL = []
inputsL.append(manualX.tolist())
#print(inputsL)

import onnxruntime as rt
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/gbr_no_mapie.onnx")
model_inv = onnx_model
session = rt.InferenceSession("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/gbr_no_mapie.onnx")
input_name = session.get_inputs()[0].name
input_data = np.array(inputsL, dtype=np.float32) 
#input_data = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
inputs = {input_name: input_data}
outputs = session.run(None, inputs)
print(outputs)

[array([[-0.14729762]], dtype=float32)]
